In [19]:
from sys import stdout

import random
import numpy as np

import matplotlib.pyplot as plt

from PIL import Image

import tensorflow as tf
from tensorflow import keras

In [21]:
!git clone https://github.com/Dilekyuksel/CapstoneProject

'git' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
!pip install gdown

import gdown

file_id = '1lSe4MBJagNzD9pFNvzaiFnaVzwDz_BIH'
gdown.download(f'https://drive.google.com/uc?id={file_id}', output='Seismic_data.sgy', quiet=False)


Defaulting to user installation because normal site-packages is not writeable


Downloading...
From (original): https://drive.google.com/uc?id=1lSe4MBJagNzD9pFNvzaiFnaVzwDz_BIH
From (redirected): https://drive.google.com/uc?id=1lSe4MBJagNzD9pFNvzaiFnaVzwDz_BIH&confirm=t&uuid=c3f4e8d3-03ae-44b9-ab31-5328a42d5a26
To: C:\Users\yasir\Jupyter-YB\0. Seismic ML\FYP Project\Seismic_data.sgy
100%|██████████| 699M/699M [00:36<00:00, 19.0MB/s] 


'Seismic_data.sgy'

In [23]:
import segyio
import numpy as np

filename = "Seismic_data.sgy"

# Step 1: Open SEG-Y with full headers, but ignore geometry
with segyio.open(filename, "r", ignore_geometry=True) as segy_file:
    segy_file.mmap()

    # Read all trace header info
    inlines = np.array(segy_file.attributes(segyio.TraceField.INLINE_3D)[:])
    crosslines = np.array(segy_file.attributes(segyio.TraceField.CROSSLINE_3D)[:])
    samples = len(segy_file.samples)

    # Find unique and sorted inline/xline numbers
    unique_ilines = np.unique(inlines)
    unique_xlines = np.unique(crosslines)

    iline_map = {iline: i for i, iline in enumerate(unique_ilines)}
    xline_map = {xline: i for i, xline in enumerate(unique_xlines)}

    # Initialize seismic cube (inline, xline, samples)
    data = np.zeros((len(unique_ilines), len(unique_xlines), samples), dtype=np.float32)

    # Step 2: Map traces to correct position
    for i in range(segy_file.tracecount):
        il = iline_map[inlines[i]]
        xl = xline_map[crosslines[i]]
        data[il, xl, :] = segy_file.trace[i]

# Step 3: Normalize
data = (data - np.min(data)) / (np.max(data) - np.min(data))

In [25]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Define interactive slider for selecting inline
inline_slider = widgets.IntSlider(
    value=100,
    min=100,
    max=500,
    step=100,
    description='Inline:',
    continuous_update=False
)

# Define function to update the plot
def plot_inline(inline_index):
    plt.figure(figsize=(12, 6))
    plt.imshow(data[inline_index, :, :].T, cmap='gray', aspect='auto')
    plt.title(f"Inline Slice {inline_index}")
    plt.axis('off')
    plt.show()

# Connect slider to plot function
widgets.interact(plot_inline, inline_index=inline_slider)



interactive(children=(IntSlider(value=100, continuous_update=False, description='Inline:', max=500, min=100, s…

<function __main__.plot_inline(inline_index)>

In [12]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Load a label mask for a given inline slice
def load_label(inline):
    label_path = f'./seismic_deep_learning/tutorial-03/mask_inline_{inline}.png'
    label = Image.open(label_path)
    label.load()
    label = np.asarray(label)[:, :, 3]  # Alpha channel
    return np.where(label > 0.5, 1, 0)

# Overlay label onto grayscale seismic image
def plot_overlay(inline):
    label = load_label(inline)
    image = data[inline, :, :].T  # Transpose so time is vertical
    tmax, xmax = image.shape

    # Create RGBA label overlay
    label_rgb = np.zeros((tmax, xmax, 4), dtype='uint8')
    label_rgb[:, :, 0] = 255
    label_rgb[:, :, 1] = 255 - 255 * label
    label_rgb[:, :, 2] = 255 - 255 * label
    label_rgb[:, :, 3] = 255 * label

    img = Image.fromarray(label_rgb, mode='RGBA')

    # Create RGBA seismic image
    image_rgb = np.zeros((tmax, xmax, 4), dtype='uint8')
    image_rgb[:, :, 0:3] = (255 * image[..., None]).astype('uint8')
    image_rgb[:, :, 3] = 255
    background = Image.fromarray(image_rgb, mode='RGBA')

    # Overlay and show
    background.paste(img, (0, 0), img)
    plt.figure(figsize=(12, 6))
    plt.imshow(background)
    plt.title(f"Inline {inline}")
    plt.axis('off')
    plt.show()

# Interactive slider
inline_slider = widgets.IntSlider(
    value=100,
    min=100,
    max=500,
    step=100,
    description='Inline:',
    continuous_update=False
)

# Display interactive plot
widgets.interact(plot_overlay, inline=inline_slider)


interactive(children=(IntSlider(value=100, continuous_update=False, description='Inline:', max=500, min=100, s…

<function __main__.plot_overlay(inline)>

In [33]:
# ---- Patch extraction function for training/validation ----
def patchify(data, label, size, number, threshold):
    """
    Extracts a specified number of (size x size) patches from seismic data and label masks.
    Only extracts patches that contain enough label signal (> threshold nonzero pixels).

    Parameters:
        data (2D array): Seismic image (inline slice)
        label (2D array): Binary label mask of same size as data
        size (int): Size of square patch
        number (int): Total number of patches to extract
        threshold (int): Minimum number of labeled pixels required in a patch

    Returns:
        X, Y: Arrays of extracted seismic patches and corresponding label patches
    """
    (t_max, x_max) = label.shape  # Dimensions of the mask
    X = np.zeros((number, size, size, 1))  # Seismic patches
    Y = np.zeros((number, size, size, 1))  # Label patches

    n = 0  # Patch counter
    while n < number:
        # Random center point inside bounds
        x = random.randint(size // 2, x_max - size // 2)
        t = random.randint(size // 2, t_max - size // 2)

        # Check if patch has enough labeled pixels
        if np.count_nonzero(label[t - size // 2 : t + size // 2,
                                  x - size // 2 : x + size // 2]) > threshold:

            # Optional transpose if dimensions don't match (data time/depth orientation fix)
            if data.shape[0] != label.shape[0]:
                data = data.T

            # Extract patch from both seismic data and label
            X[n, :, :, 0] = data[t - size // 2 : t + size // 2,
                                 x - size // 2 : x + size // 2]
            Y[n, :, :, 0] = label[t - size // 2 : t + size // 2,
                                  x - size // 2 : x + size // 2]
            n += 1

    return X, Y


In [14]:
# --- Patch size and dataset split ---
size       = 64       # Patch size: 64x64
threshold  = 0        # No label filtering; include all patches
num_train  = 8000     # Number of training patches
num_val    = 2000     # Number of validation patches

# --- Extract validation patches from inline 300 ---
X_val = np.zeros((num_val, size, size, 1))
Y_val = np.zeros((num_val, size, size, 1))

label   = load_label(300)     # Load binary label mask
seismic = data[300, ...]      # Get seismic image at inline 300
X_val, Y_val = patchify(seismic, label, size, num_val, threshold)

# --- Extract training patches from inlines 100, 200, 400, 500 ---
X_train = np.zeros((num_train, size, size, 1))
Y_train = np.zeros((num_train, size, size, 1))

n = 0  # Patch counter
for inline in [100, 200, 400, 500]:
    label   = load_label(inline)     # Load label for current inline
    seismic = data[inline, ...]      # Get seismic data for current inline

    # Extract equal number of patches per inline
    X_part, Y_part = patchify(seismic, label, size, num_train // 4, threshold)

    # Store patches into training arrays
    X_train[n : n + num_train // 4, ...] = X_part
    Y_train[n : n + num_train // 4, ...] = Y_part
    n += num_train // 4

FileNotFoundError: [Errno 2] No such file or directory: './seismic_deep_learning/tutorial-03/mask_inline_300.png'

In [ ]:
# Create a 2-row by 10-column grid of subplots
# First row: seismic patches (X_train), second row: label masks (Y_train)
fig, axs = plt.subplots(2, 10, figsize=(15, 3))

k = 0  # Patch index

# --- Plot 10 seismic image patches ---
for m in range(10):
    axs[0, m].imshow(X_train[k, :, :, 0],  # Plot grayscale patch
                     interpolation='spline16',
                     cmap=plt.cm.gray,
                     aspect=1)
    axs[0, m].set_xticks([])
    axs[0, m].set_yticks([])
    k += 1

k = 0  # Reset index

# --- Plot the corresponding 10 label masks ---
for m in range(10):
    axs[1, m].imshow(Y_train[k, :, :, 0],  # Plot label mask (binary)
                     interpolation='spline16',
                     aspect=1)
    axs[1, m].set_xticks([])
    axs[1, m].set_yticks([])
    k += 1



In [ ]:
import tensorflow as tf

# ---- Encoder block: Downsampling ----
def down_block(x, filters, kernel_size=(3, 3), padding="same", strides=1):
    c = tf.keras.layers.Conv2D(filters, kernel_size, padding=padding, strides=strides, activation="relu")(x)
    c = tf.keras.layers.Conv2D(filters, kernel_size, padding=padding, strides=strides, activation="relu")(c)
    p = tf.keras.layers.MaxPool2D((2, 2), (2, 2))(c)
    return c, p

# ---- Decoder block: Upsampling ----
def up_block(x, skip, filters, kernel_size=(3, 3), padding="same", strides=1):
    us = tf.keras.layers.UpSampling2D((2, 2))(x)
    concat = tf.keras.layers.Concatenate()([us, skip])
    c = tf.keras.layers.Conv2D(filters, kernel_size, padding=padding, strides=strides, activation="relu")(concat)
    c = tf.keras.layers.Conv2D(filters, kernel_size, padding=padding, strides=strides, activation="relu")(c)
    return c

# ---- Bottleneck block ----
def bottleneck(x, filters, kernel_size=(3, 3), padding="same", strides=1):
    c = tf.keras.layers.Conv2D(filters, kernel_size, padding=padding, strides=strides, activation="relu")(x)
    c = tf.keras.layers.Conv2D(filters, kernel_size, padding=padding, strides=strides, activation="relu")(c)
    return c

# ---- Full UNet architecture ----
def UNet(input_size):
    f = [16, 32, 64, 128, 256]  # Number of filters at each level

    inputs = tf.keras.layers.Input((input_size, input_size, 1))

    # Encoder (contracting path)
    c1, p1 = down_block(inputs, f[0])  # 64 -> 32
    c2, p2 = down_block(p1, f[1])      # 32 -> 16
    c3, p3 = down_block(p2, f[2])      # 16 -> 8
    c4, p4 = down_block(p3, f[3])      # 8 -> 4

    # Bottleneck
    bn = bottleneck(p4, f[4])          # 4 -> 4

    # Decoder (expanding path)
    u1 = up_block(bn, c4, f[3])        # 4 -> 8
    u2 = up_block(u1, c3, f[2])        # 8 -> 16
    u3 = up_block(u2, c2, f[1])        # 16 -> 32
    u4 = up_block(u3, c1, f[0])        # 32 -> 64

    # Output layer: 1 channel with sigmoid for binary segmentation
    outputs = tf.keras.layers.Conv2D(1, (1, 1), padding="same", activation="sigmoid")(u4)

    model = tf.keras.models.Model(inputs, outputs)
    return model

# ---- Instantiate and compile the model ----
size = 64  # Example input patch size
model = UNet(size)

model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])  # Or use "acc" for shorthand


In [ ]:
history = model.fit(X_train,
                    Y_train,
                    validation_data=(X_val, Y_val),
                    epochs=5)

In [ ]:
# --- Plot accuracy over training epochs ---
# NOTE: Change 'acc' to 'accuracy' if needed
plt.plot(history.history['accuracy'])           # Training accuracy
plt.plot(history.history['val_accuracy'])       # Validation accuracy
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

# --- Plot loss over training epochs ---
plt.plot(history.history['loss'])               # Training loss
plt.plot(history.history['val_loss'])           # Validation loss
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

In [ ]:
!pip install scikit-learn

from sklearn.metrics import confusion_matrix

# Predict on validation set
Y_pred = model.predict(X_val)
Y_true = Y_val

# Initialize total confusion matrix for binary classification: [TN, FP], [FN, TP]
cm = np.zeros((2, 2))

# Loop through each patch in the validation set
for n in range(Y_true.shape[0]):
    # Flatten both prediction and ground truth masks to 1D
    # Round predictions to binary values (0 or 1)
    y_true = Y_true[n, :, :, 0].round().flatten().astype(int)
    y_pred = Y_pred[n, :, :, 0].round().flatten().astype(int)

    # Compute confusion matrix for the current patch
    cm_batch = confusion_matrix(y_true, y_pred, labels=[0, 1])

    # Accumulate results
    cm += cm_batch


# --- Plot styling ---
cmap = plt.cm.Blues         # Color map for the heatmap
normalize = True            # Whether to normalize the matrix
title = 'Normalized confusion matrix'
classes = ['No fault', 'Fault']  # Class labels

# --- Normalize the confusion matrix by row (true label) ---
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# --- Create the figure and heatmap ---
fig, ax = plt.subplots()
im = ax.imshow(cm_norm, interpolation='nearest', cmap=cmap)
ax.figure.colorbar(im, ax=ax)  # Add color scale bar

# --- Configure axis ticks and labels ---
ax.set(
    xticks=np.arange(cm_norm.shape[1]),
    yticks=np.arange(cm_norm.shape[0]),
    xticklabels=classes,
    yticklabels=classes,
    title=title,
    ylabel='True label',
    xlabel='Predicted label'
)

# --- Rotate x-axis labels for better readability ---
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# --- Add numerical values inside each cell ---
fmt = '.2f' if normalize else 'd'
thresh = cm_norm.max() / 2.  # Threshold for text color contrast
for i in range(cm_norm.shape[0]):
    for j in range(cm_norm.shape[1]):
        ax.text(j, i, format(cm_norm[i, j], fmt),
                ha="center", va="center",
                color="white" if cm_norm[i, j] > thresh else "black")

# --- Final layout adjustment ---
fig.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# ---- Tile seismic image into overlapping patches ----
def tile(data, wsize, dt, dx):
    """
    Tiles the 2D seismic data into overlapping patches with stride (dt, dx).
    Handles edges by repeating the border areas.
    """
    t_max, x_max = data.shape

    n_patch = (t_max // dt + 1) * (x_max // dx + 1)
    data_patch = np.zeros((n_patch, wsize, wsize, 1))

    n = 0
    for t in range(0, t_max, dt):
        for x in range(0, x_max, dx):
            if t_max - t < wsize and x_max - x < wsize:
                data_patch[n, :, :, 0] = data[t_max - wsize:t_max, x_max - wsize:x_max]
            elif x_max - x < wsize:
                data_patch[n, :, :, 0] = data[t:t + wsize, x_max - wsize:x_max]
            elif t_max - t < wsize:
                data_patch[n, :, :, 0] = data[t_max - wsize:t_max, x:x + wsize]
            else:
                data_patch[n, :, :, 0] = data[t:t + wsize, x:x + wsize]
            n += 1

    return data_patch

# ---- Merge prediction patches ----
def merge(data_patch, t_max, x_max, wsize, dt, dx):
    """
    Merges prediction patches back into a single 2D result.
    Uses a count matrix to average overlapping predictions.
    """
    data_new = np.zeros((t_max, x_max, 1))
    count = np.zeros((t_max, x_max, 1))
    n = 0
    for t in range(0, t_max, dt):
        for x in range(0, x_max, dx):
            if t_max - t < wsize and x_max - x < wsize:
                data_new[t_max - wsize:t_max, x_max - wsize:x_max, 0] += data_patch[n, :, :, 0]
                count[t_max - wsize:t_max, x_max - wsize:x_max, 0] += 1
            elif x_max - x < wsize:
                data_new[t:t + wsize, x_max - wsize:x_max, 0] += data_patch[n, :, :, 0]
                count[t:t + wsize, x_max - wsize:x_max, 0] += 1
            elif t_max - t < wsize:
                data_new[t_max - wsize:t_max, x:x + wsize, 0] += data_patch[n, :, :, 0]
                count[t_max - wsize:t_max, x:x + wsize, 0] += 1
            else:
                data_new[t:t + wsize, x:x + wsize, 0] += data_patch[n, :, :, 0]
                count[t:t + wsize, x:x + wsize, 0] += 1
            n += 1

    return data_new / count  # Average overlapping predictions

# ---- Parameters ----
dt = 16
dx = 16
size = 64  # Patch size used in training

# Get dimensions (data is assumed to be shape: (inline, time, xline))
_, t_max, x_max = data.shape

# ---- Plot overlay ----
def plot_overlay(image, mask):
   
    tmax, xmax = image.shape
    label_rgb = np.zeros((tmax, xmax, 4), dtype='uint8')
    label_rgb[:, :, 0] = 255
    label_rgb[:, :, 1] = 255 - 255 * mask
    label_rgb[:, :, 2] = 255 - 255 * mask
    label_rgb[:, :, 3] = (255 * mask).astype('uint8')

    from PIL import Image
    label_img = Image.fromarray(label_rgb, mode='RGBA')
    image_rgb = np.zeros((tmax, xmax, 4), dtype='uint8')
    image_rgb[:, :, 0:3] = (255 * image[..., None]).astype('uint8')
    image_rgb[:, :, 3] = 255
    background = Image.fromarray(image_rgb, mode='RGBA')
    background.paste(label_img, (0, 0), label_img)

    plt.imshow(background)
    plt.axis('off')
    plt.show()

# ---- Interactive inline slider ----
def predict_and_show(inline):
    inline_data = data[inline, :, :].T
    data_tiles = tile(inline_data, size, dt, dx)
    result_tiles = model.predict_on_batch(data_tiles)
    result = merge(result_tiles, *inline_data.shape, size, dt, dx)
    plot_overlay(inline_data, result[:, :, 0])

# Set up slider widget
slider = widgets.IntSlider(value=100, min=100, max=500, step=100, description='Inline:')
widgets.interact(predict_and_show, inline=slider)


In [ ]:
# ---- Choose the depth (time index) of the seismic volume to visualize ----
depth = 420  # This selects the 420th time/depth sample

# ---- Extract a 2D horizontal slice from the 3D seismic cube ----
# Shape of `data` is assumed to be (inline, crossline, time)
# This grabs the 2D section at a fixed time across all inlines and crosslines
time_slice = data[:, :, depth]

# ---- Plot the extracted time slice as a grayscale image ----
plt.figure(figsize=(10, 10))                          # Set figure size
plt.imshow(time_slice, cmap=plt.cm.gray)              # Display the slice with grayscale colormap
plt.axis('off')                                       # Hide axis ticks for a cleaner view
plt.title(f'Seismic Time Slice at Depth {depth}')     # Optional: add title
plt.show()                                            # Render the plot

In [ ]:
import sys

# ---- Parameters ----
depth = 420              # Time/depth slice index
size = 64                # Patch size (must match training)
batch_size = 1000        # Inference batch size

# ---- Prepare prediction storage ----
xmax, ymax = data.shape[0], data.shape[1]          # Inline × Crossline
nexp = (xmax - 2 * size) * (ymax - 2 * size)       # Total number of prediction points

coord = np.zeros((batch_size, 2), int)             # Coordinate buffer
# Change patch shape to store 2D data (batch_size, size, size, 1)
patch = np.zeros((batch_size, size, size, 1))      # Patch buffer
prediction = np.zeros((xmax, ymax), dtype=np.float32)  # Prediction map

# ---- Sliding window prediction ----
m = 0
n_batch = 0

for x in range(size, xmax - size):
    for y in range(size, ymax - size):
        # Extracting 2D patch from data at the specified depth
        patch[m, :, :, 0] = data[
            x - size // 2 : x + size // 2,
            y - size // 2 : y + size // 2,
            depth
        ]
        coord[m, :] = [x, y]
        m += 1

        if m == batch_size:
            # Predict on the 2D patches
            preds = model.predict_on_batch(patch)[:, size // 2, size // 2, 0]
            prediction[coord[:, 0], coord[:, 1]] = preds
            n_batch += 1
            sys.stdout.write(f"\rProcessing batch {n_batch} of {round(nexp / batch_size)}")
            sys.stdout.flush()
            m = 0

# ---- Final batch (if incomplete) ----
if m > 0:
    preds = model.predict_on_batch(patch[:m])[:, size // 2, size // 2, 0]
    prediction[coord[:m, 0], coord[:m, 1]] = preds
    sys.stdout.write(f"\rProcessing batch {n_batch + 1} of {round(nexp / batch_size)}\n")
    sys.stdout.flush()

# ---- Extract time slice for display ----
# Note: no need to transpose here, data is (inline, crossline, time)
time_slice = data[:, :, depth]

# ---- Plot prediction over seismic time slice ----
plt.figure(figsize=(10, 10))

# ---- Plot overlay using function defined earlier
plot_overlay(time_slice, prediction)

plt.title(f'Prediction Overlay at Time Slice {depth}')
plt.axis('off')
plt.show()